## Abstraction Layer

- It hides complex, messy underlying code behind a clean, simple, and uniform interface. 

- It allows you to interact with multiple different systems using the exact same commands, without caring how those systems work under the hood.

- Example:

    - The Abstraction Layer (The Standardized Interface):
    
        - Imagine you want your app to support three different AI companies: OpenAI (gpt-4o), Anthropic (claude-3-5-sonnet), and Google (gemini-1-5-pro). 
    
        - Normally, each company forces you to use their unique code library, unique authentication methods, and unique parameter formats.
    
        - Instead of writing three separate blocks of messy code, you use an abstraction layer like LiteLLM or OpenRouter.

### Concept of Abstraction can be clearly illustrated using:

1. OpenRouter

2. LiteLLM

3. LangChain

In [ ]:
%pip install litellm
%pip install langchain_google_genai

In [4]:
import os
from dotenv import load_dotenv
from IPython.display import display, Markdown
from openai import OpenAI

load_dotenv(override=True)  # load .env into os.environ so libraries that read env vars (like litellm or LangChain) can find keys like GEMINI_API_KEY

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openrouter_base_url = os.getenv('OPENROUTER_BASE_URL')

if openrouter_api_key:
    print(f'OpenRouter API key found and starts with "{openrouter_api_key[:3]}')
else:
    print('OpenRouter API key not found')

OpenRouter API key found and starts with "sk-


In [5]:
openrouter = OpenAI(base_url=openrouter_base_url, api_key=openrouter_api_key)

### OpenRouter.AI

- OpenRouter is an API aggregator and router built specifically for Large Language Models [LLMs]

- It acts as a single, unified gateway that connects developers to hundreds of different AI models from various providers

- Instead of managing separate accounts and API keys for OpenAI, Anthropic, Google, and Meta, OpenRouter gives you a single unified endpoint and key to access hundreds of open and closed-source models.

#### Why it's useful

1. Provides a Single Unified API:

    - Instead of installing separate software kits for OpenAI, Anthropic, Google, and Mistral, OpenRouter provides one single API key and code format. You can swap out the model name in your code to change providers instantly without rewriting your application.

2. Acts as an Intelligent Router:

    - OpenRouter doesn't just list models; it dynamically manages your traffic.
    
    - Automatic Fallbacks: If a specific host or provider goes down, OpenRouter automatically routes your request to a backup server running the same model to ensure your app stays online.
    
    - Price and Speed Optimization: For open-source models hosted by multiple companies (like Groq, Lepton, or Together AI), OpenRouter can automatically route your request to whichever host is cheapest or fastest at that exact millisecond.

3. Hosts Over 400+ Models

    - It offers access to almost every major model in existence, including:
    
    - Proprietary Models: GPT-4o, Claude 3.5 Sonnet, and Gemini Pro.
    
    - Open-Source Models: DeepSeek R1, Llama 3.3, Qwen, and Mistral.
    
    - Free Models: It constantly maintains a selection of over 50 completely free, rate-limited open-source models for prototyping.


In [6]:
tell_me_a_joke = [
    {'role':'system', 'content':'You are a helpful and humorous assistant that always tries to make people smile'},
    {'role': 'user', 'content':'Tell me a funny joke'}
]

In [7]:
response = openrouter.chat.completions.create(
    model = 'nvidia/nemotron-3-ultra-550b-a55b:free',               # Nemotron is an open-source model from Nvidia
    messages=tell_me_a_joke
)                                                                   # openrouter provides a single unified API, so we can call any listed model through this API

display(Markdown(response.choices[0].message.content))

I told my wife she was drawing her eyebrows too high.

She looked surprised. 😲

### LiteLLM

- LiteLLM is a lightweight code level abstraction layer that provides a common interface for interacting with multiple LLM providers

- While OpenRouter is an online service, LiteLLM is a library you install (pip install litellm) in your codebase.

- It standardizes code calls across 100+ LLM APIs (OpenAI, Azure, Bedrock, Vertex AI, Ollama, Hugging Face). 

- You call litellm.completion(model="...", messages=[...]), and LiteLLM translates your input into whatever custom payload structure that specific cloud provider expects, then normalizes the response back into OpenAI's standard format.

#### Why it’s useful

- Vendor Lock-in Insurance: 

    - Switch from OpenAI to AWS Bedrock or an in-house Ollama model by changing just a string name in code.

- Self-Hosted Control: 

    - Keeps API routing inside your own infrastructure/VPC (unlike OpenRouter, which is a third-party server).

- Enterprise Features: 

    - Handles retries, fallback models, rate-limiting, and cost tracking out-of-the-box.

In [11]:
# %pip install litellm

from litellm import completion

response = completion(
    model = 'gemini/gemini-3.6-flash',                      # we can change the model to any available model but we have to make sure that respective API key is exposed to the "litellm" via os
    messages = tell_me_a_joke,
    fallbacks = ['nvidia/nemotron-3-ultra-550b-a55b:free']  # automatic fallback to nvidia's nemotron if gemini is down
)

display(Markdown(response.choices[0].message.content))

Here’s one of my absolute favorites:

My wife told me to stop impersonating a flamingo...

...so I had to put my foot down! 🦩

*(I hope that gave you a chuckle! If not, I’m winging it here!)*

### LangChain

- LangChain is an open-source development framework designed to help software engineers to build applications powered by Large Language Models [LLMs]

- LangChain operates at the highest abstraction layer (Application & Orchestration). It isn't just about calling an LLM—it’s about chaining LLMs together with external memory, vector databases (RAG), and external tools (web search, calculators, code execution).

- It simplifies the entire development by stitching together prompts, models, vector databases and tools into automated workflows or "chains"

### Why it's useful

- Component Chaining:

    - It allows you to link multiple independent LLM tasks together. For example, Chain A can search the web, Chain B can summarize the findings, and Chain C can format the output into an email.

- Memory Management:

    - Out of the box, LLMs are completely stateless—they do not remember past messages. 
    
    - LangChain injects automated memory systems into your scripts to retain conversation history across long web sessions.

- Data Ingestion [RAG]:

    - It provides streamlined connectors to read, split, and index local PDFs, databases, or documentation websites into text vector formats, making it easy to build a "chat with your data" chatbot.

In [9]:
# %pip install langchain_google_genai

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model = 'gemini-3.5-flash')
response = llm.invoke(tell_me_a_joke)

display(Markdown(response.text))


I would love to! This is one of my absolute favorites. Get ready, because this one is a real "step" in the right direction to make you smile...

***

My wife told me to stop impersonating a flamingo. 

...I had to put my foot down! 

***

*Badum-tsss!* 🥁 

Hopefully, that gave you a little lift—or at least kept you standing! Let me know if you want another one, I’ve got a million of 'em!

In [12]:
# the true use case of LangChain - a heavy weight abstraction layer mostly used for orchestration [chaining LLMs together]

''' 
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

# 1. Define the orchestration components
llm = ChatAnthropic(model="claude-3-5-sonnet-20240620")
tools = [DuckDuckGoSearchRun()]
prompt = hub.pull("hwchase17/react")

# 2. Build the agent pipeline
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools)

# 3. Execute a multi-step task (LLM decides when to search, reads results, then answers)
agent_executor.invoke({"input": "Search the web for today's top tech news and summarize it."})
'''

' \nfrom langchain_community.tools import DuckDuckGoSearchRun\nfrom langchain_anthropic import ChatAnthropic\nfrom langchain.agents import create_react_agent, AgentExecutor\nfrom langchain import hub\n\n# 1. Define the orchestration components\nllm = ChatAnthropic(model="claude-3-5-sonnet-20240620")\ntools = [DuckDuckGoSearchRun()]\nprompt = hub.pull("hwchase17/react")\n\n# 2. Build the agent pipeline\nagent = create_react_agent(llm, tools, prompt)\nagent_executor = AgentExecutor(agent=agent, tools=tools)\n\n# 3. Execute a multi-step task (LLM decides when to search, reads results, then answers)\nagent_executor.invoke({"input": "Search the web for today\'s top tech news and summarize it."})\n'

## The Core Difference

- LiteLLM is a Low-Level Model Abstraction Layer: 

    - Its only job is to normalize the API input and output formats. It takes the varying API schemas of OpenAI, Anthropic, Gemini, or Groq and forces them to use a single, identical completion() function. It does not care about your application logic, prompts, or databases.
    
- LangChain is a High-Level Orchestration Abstraction Layer: 

    - It abstracts everything around the model. It provides uniform interfaces for things that have nothing to do with raw API calls, such as vector databases (Chroma, Pinecone), document loaders (PDFs, URLs), long-term conversational memory, and prompt templates.

## How They Work Together

- Because they solve different problems, they are frequently used together. 

- In an enterprise system, you can use LangChain to handle your chunking, vector database searches, and memory management, and then plug LiteLLM into LangChain as the underlying execution engine to call the models.

In [ ]:
''' 
#### How They Work Together

These tools are not mutually exclusive—you can combine all three in a single architecture:

     [ Your Application ]
          ↓
     [ LangChain ]        <-- Handles logic: "Search web -> Read DB -> Generate answer"
          ↓
     [ LiteLLM ]         <-- Handles code abstraction: Normalizes request format & fallbacks
          ↓
     [ OpenRouter ]       <-- Handles cloud routing: Sends call to the cheapest provider
          ↓
     [ OpenAI / Anthropic / Meta ] <-- The actual LLM generates tokens

'''